# Data Cleaning and Validation

This notebook prepares the raw market data collected in the ingestion stage for consistent use across the event analyses.

The project combines data from several independent sources, and each source uses a different structure, timestamp format and market convention.

The cleaning process therefore focuses on:

- standardizing schemas and column names;
- converting timestamps and numeric fields into consistent data types;
- checking for missing values and duplicate observations;
- identifying gaps in hourly market coverage;
- documenting source-specific limitations before analysis begins.

The original raw files are preserved unchanged. Cleaned datasets are saved separately for downstream analysis.

No event interpretation or risk conclusions are produced at this stage.

In [1]:
import pandas as pd

## 1. Daily Market Data — Initial Quality Check

The first dataset contains daily BTC, ETH, USDT and USDC market data collected from yfinance.

Before cleaning, the raw file is inspected to understand its structure and identify any issues that could affect later analysis.

The initial checks include:

- dataset dimensions;
- column structure;
- missing values;
- available date range;
- data types.

In [2]:
crypto_daily_raw = pd.read_csv("../data/raw/daily/crypto_daily_full.csv", header=[0, 1])
crypto_daily_raw.head()

Price    Adj Close                                        Close  \
       Ticker      BTC-USD     ETH-USD  USDC-USD  USDT-USD      BTC-USD   
0        Date          NaN         NaN       NaN       NaN          NaN   
1  2020-01-01  7200.174316  130.802002  1.004079  0.999836  7200.174316   
2  2020-01-02  6985.470215  127.410179  1.005017  1.001565  6985.470215   
3  2020-01-03  7344.884277  134.171707  1.005273  1.004192  7344.884277   
4  2020-01-04  7410.656738  135.069366  1.009466  1.007472  7410.656738   

                                          High  ...       Low            \
      ETH-USD  USDC-USD  USDT-USD      BTC-USD  ...  USDC-USD  USDT-USD   
0         NaN       NaN       NaN          NaN  ...       NaN       NaN   
1  130.802002  1.004079  0.999836  7254.330566  ...  1.001989  0.994924   
2  127.410179  1.005017  1.001565  7212.155273  ...  1.002543  0.986515   
3  134.171707  1.005273  1.004192  7413.715332  ...  0.988455  0.988027   
4  135.069366  1.009466  1.007472  7427.385742  ...  1.001273  0.999160   

          Open                                        Volume                \
       BTC-USD     ETH-USD  USDC-USD  USDT-USD       BTC-USD       ETH-USD   
0          NaN         NaN       NaN       NaN           NaN           NaN   
1  7194.892090  129.630661  1.003730  0.999571  1.856566e+10  7.935230e+09   
2  7202.551270  130.820038  1.004431  0.999788  2.080208e+10  8.032709e+09   
3  6984.428711  127.411263  1.005357  1.001183  2.811148e+10  1.047685e+10   
4  7345.375488  134.168518  1.004818  1.003510  1.844427e+10  7.430905e+09   

                              
      USDC-USD      USDT-USD  
0          NaN           NaN  
1  242586528.0  2.150314e+10  
2  318268134.0  2.421231e+10  
3  374792167.0  3.242029e+10  
4  334494308.0  2.158563e+10  

[5 rows x 25 columns]

In [3]:
crypto_daily_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2424 entries, 0 to 2423
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   (Price, Ticker)        2424 non-null   str    
 1   (Adj Close, BTC-USD)   2423 non-null   float64
 2   (Adj Close, ETH-USD)   2423 non-null   float64
 3   (Adj Close, USDC-USD)  2423 non-null   float64
 4   (Adj Close, USDT-USD)  2423 non-null   float64
 5   (Close, BTC-USD)       2423 non-null   float64
 6   (Close, ETH-USD)       2423 non-null   float64
 7   (Close, USDC-USD)      2423 non-null   float64
 8   (Close, USDT-USD)      2423 non-null   float64
 9   (High, BTC-USD)        2423 non-null   float64
 10  (High, ETH-USD)        2423 non-null   float64
 11  (High, USDC-USD)       2423 non-null   float64
 12  (High, USDT-USD)       2423 non-null   float64
 13  (Low, BTC-USD)         2423 non-null   float64
 14  (Low, ETH-USD)         2423 non-null   float64
 15  (Low, USDC-USD)

In [4]:
crypto_daily_raw.shape

(2424, 25)

In [5]:
crypto_daily_raw.isna().sum()

Price      Ticker      0
Adj Close  BTC-USD     1
           ETH-USD     1
           USDC-USD    1
           USDT-USD    1
Close      BTC-USD     1
           ETH-USD     1
           USDC-USD    1
           USDT-USD    1
High       BTC-USD     1
           ETH-USD     1
           USDC-USD    1
           USDT-USD    1
Low        BTC-USD     1
           ETH-USD     1
           USDC-USD    1
           USDT-USD    1
Open       BTC-USD     1
           ETH-USD     1
           USDC-USD    1
           USDT-USD    1
Volume     BTC-USD     1
           ETH-USD     1
           USDC-USD    1
           USDT-USD    1
dtype: int64

### Finding

The raw CSV preserves the original multi-level column structure returned by yfinance.

It also contains an additional row with the value `Date`, which causes:

- one apparent missing value in several fields;
- market-value columns to be interpreted as text instead of numeric data.

This is a formatting issue introduced by the raw CSV structure rather than missing market information.

The extra row is removed during cleaning, after which the schema and data types can be standardized.

## 2. Daily Data Cleaning and Schema Standardization

The raw dataset is kept unchanged for traceability.

A separate processed dataset is created for analysis. The cleaning steps include:

- removing the extra header-like row;
- replacing the yfinance multi-level columns with clear `snake_case` names;
- converting the date field to datetime;
- converting market fields to numeric types;
- checking for missing values and duplicate dates.

In [6]:
crypto_daily_clean = crypto_daily_raw.copy()

In [7]:
crypto_daily_clean = crypto_daily_clean[crypto_daily_clean[("Price", "Ticker")] != "Date"].reset_index(drop=True)
crypto_daily_clean.head()

Price    Adj Close                                        Close  \
       Ticker      BTC-USD     ETH-USD  USDC-USD  USDT-USD      BTC-USD   
0  2020-01-01  7200.174316  130.802002  1.004079  0.999836  7200.174316   
1  2020-01-02  6985.470215  127.410179  1.005017  1.001565  6985.470215   
2  2020-01-03  7344.884277  134.171707  1.005273  1.004192  7344.884277   
3  2020-01-04  7410.656738  135.069366  1.009466  1.007472  7410.656738   
4  2020-01-05  7411.317383  136.276779  1.008497  1.006197  7411.317383   

                                          High  ...       Low            \
      ETH-USD  USDC-USD  USDT-USD      BTC-USD  ...  USDC-USD  USDT-USD   
0  130.802002  1.004079  0.999836  7254.330566  ...  1.001989  0.994924   
1  127.410179  1.005017  1.001565  7212.155273  ...  1.002543  0.986515   
2  134.171707  1.005273  1.004192  7413.715332  ...  0.988455  0.988027   
3  135.069366  1.009466  1.007472  7427.385742  ...  1.001273  0.999160   
4  136.276779  1.008497  1.006197  7544.497070  ...  1.003932  1.001758   

          Open                                        Volume                \
       BTC-USD     ETH-USD  USDC-USD  USDT-USD       BTC-USD       ETH-USD   
0  7194.892090  129.630661  1.003730  0.999571  1.856566e+10  7.935230e+09   
1  7202.551270  130.820038  1.004431  0.999788  2.080208e+10  8.032709e+09   
2  6984.428711  127.411263  1.005357  1.001183  2.811148e+10  1.047685e+10   
3  7345.375488  134.168518  1.004818  1.003510  1.844427e+10  7.430905e+09   
4  7410.451660  135.072098  1.008748  1.009921  1.972507e+10  7.526675e+09   

                              
      USDC-USD      USDT-USD  
0  242586528.0  2.150314e+10  
1  318268134.0  2.421231e+10  
2  374792167.0  3.242029e+10  
3  334494308.0  2.158563e+10  
4  334597544.0  2.409014e+10  

[5 rows x 25 columns]

In [8]:
new_columns = []

for metric, ticker in crypto_daily_clean.columns:
    if metric == "Price":
        new_columns.append("date")
    else:
        asset = ticker.replace("-USD", "").lower()
        metric = metric.replace(" ", "_").lower()
        new_columns.append(f"{asset}_{metric}")

crypto_daily_clean.columns = new_columns

crypto_daily_clean.head()

,date,btc_adj_close,eth_adj_close,usdc_adj_close,usdt_adj_close,btc_close,eth_close,usdc_close,usdt_close,btc_high,...,usdc_low,usdt_low,btc_open,eth_open,usdc_open,usdt_open,btc_volume,eth_volume,usdc_volume,usdt_volume
0,2020-01-01,7200.174316,130.802002,1.004079,0.999836,7200.174316,130.802002,1.004079,0.999836,7254.330566,...,1.001989,0.994924,7194.892090,129.630661,1.003730,0.999571,1.856566e+10,7.935230e+09,242586528.0,2.150314e+10
1,2020-01-02,6985.470215,127.410179,1.005017,1.001565,6985.470215,127.410179,1.005017,1.001565,7212.155273,...,1.002543,0.986515,7202.551270,130.820038,1.004431,0.999788,2.080208e+10,8.032709e+09,318268134.0,2.421231e+10
2,2020-01-03,7344.884277,134.171707,1.005273,1.004192,7344.884277,134.171707,1.005273,1.004192,7413.715332,...,0.988455,0.988027,6984.428711,127.411263,1.005357,1.001183,2.811148e+10,1.047685e+10,374792167.0,3.242029e+10
3,2020-01-04,7410.656738,135.069366,1.009466,1.007472,7410.656738,135.069366,1.009466,1.007472,7427.385742,...,1.001273,0.999160,7345.375488,134.168518,1.004818,1.003510,1.844427e+10,7.430905e+09,334494308.0,2.158563e+10
4,2020-01-05,7411.317383,136.276779,1.008497,1.006197,7411.317383,136.276779,1.008497,1.006197,7544.497070,...,1.003932,1.001758,7410.451660,135.072098,1.008748,1.009921,1.972507e+10,7.526675e+09,334597544.0,2.409014e+10


In [9]:
crypto_daily_clean["date"] = pd.to_datetime(crypto_daily_clean["date"])
crypto_daily_clean.head()

,date,btc_adj_close,eth_adj_close,usdc_adj_close,usdt_adj_close,btc_close,eth_close,usdc_close,usdt_close,btc_high,...,usdc_low,usdt_low,btc_open,eth_open,usdc_open,usdt_open,btc_volume,eth_volume,usdc_volume,usdt_volume
0,2020-01-01,7200.174316,130.802002,1.004079,0.999836,7200.174316,130.802002,1.004079,0.999836,7254.330566,...,1.001989,0.994924,7194.892090,129.630661,1.003730,0.999571,1.856566e+10,7.935230e+09,242586528.0,2.150314e+10
1,2020-01-02,6985.470215,127.410179,1.005017,1.001565,6985.470215,127.410179,1.005017,1.001565,7212.155273,...,1.002543,0.986515,7202.551270,130.820038,1.004431,0.999788,2.080208e+10,8.032709e+09,318268134.0,2.421231e+10
2,2020-01-03,7344.884277,134.171707,1.005273,1.004192,7344.884277,134.171707,1.005273,1.004192,7413.715332,...,0.988455,0.988027,6984.428711,127.411263,1.005357,1.001183,2.811148e+10,1.047685e+10,374792167.0,3.242029e+10
3,2020-01-04,7410.656738,135.069366,1.009466,1.007472,7410.656738,135.069366,1.009466,1.007472,7427.385742,...,1.001273,0.999160,7345.375488,134.168518,1.004818,1.003510,1.844427e+10,7.430905e+09,334494308.0,2.158563e+10
4,2020-01-05,7411.317383,136.276779,1.008497,1.006197,7411.317383,136.276779,1.008497,1.006197,7544.497070,...,1.003932,1.001758,7410.451660,135.072098,1.008748,1.009921,1.972507e+10,7.526675e+09,334597544.0,2.409014e+10


In [10]:
crypto_daily_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 2423 entries, 0 to 2422
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date            2423 non-null   datetime64[us]
 1   btc_adj_close   2423 non-null   float64       
 2   eth_adj_close   2423 non-null   float64       
 3   usdc_adj_close  2423 non-null   float64       
 4   usdt_adj_close  2423 non-null   float64       
 5   btc_close       2423 non-null   float64       
 6   eth_close       2423 non-null   float64       
 7   usdc_close      2423 non-null   float64       
 8   usdt_close      2423 non-null   float64       
 9   btc_high        2423 non-null   float64       
 10  eth_high        2423 non-null   float64       
 11  usdc_high       2423 non-null   float64       
 12  usdt_high       2423 non-null   float64       
 13  btc_low         2423 non-null   float64       
 14  eth_low         2423 non-null   float64       
 15  usdc_low       

In [11]:
crypto_daily_clean.isna().sum()


date              0
btc_adj_close     0
eth_adj_close     0
usdc_adj_close    0
usdt_adj_close    0
btc_close         0
eth_close         0
usdc_close        0
usdt_close        0
btc_high          0
eth_high          0
usdc_high         0
usdt_high         0
btc_low           0
eth_low           0
usdc_low          0
usdt_low          0
btc_open          0
eth_open          0
usdc_open         0
usdt_open         0
btc_volume        0
eth_volume        0
usdc_volume       0
usdt_volume       0
dtype: int64

In [12]:
crypto_daily_clean["date"].min(), crypto_daily_clean["date"].max()

(Timestamp('2020-01-01 00:00:00'), Timestamp('2026-08-19 00:00:00'))

In [13]:
crypto_daily_clean.duplicated(subset=["date"]).sum()

np.int64(0)

In [14]:
(crypto_daily_clean["btc_adj_close"] == crypto_daily_clean["btc_close"]).all()

np.True_

In [15]:
(crypto_daily_clean["eth_adj_close"] == crypto_daily_clean["eth_close"]).all()

np.True_

In [16]:
(crypto_daily_clean["usdt_adj_close"] == crypto_daily_clean["usdt_close"]).all()

np.True_

In [17]:
(crypto_daily_clean["usdc_adj_close"] == crypto_daily_clean["usdc_close"]).all()

np.True_

### Redundant Feature Check — Adjusted Close

yfinance provides both `Close` and `Adj Close` prices.

`Adjusted Close` is normally used when historical prices need to account for events such as stock splits or dividends.

For BTC, ETH, USDT and USDC in this dataset, the adjusted and unadjusted closing prices are identical across all observations.

Because `Adj Close` does not add new information here, it is removed from the processed dataset and `Close` is retained for analysis.

In [18]:
crypto_daily_clean = crypto_daily_clean.drop(
    columns=[
        "btc_adj_close",
        "eth_adj_close",
        "usdt_adj_close",
        "usdc_adj_close"
    ]
)

In [19]:
crypto_daily_clean.columns

Index(['date', 'btc_close', 'eth_close', 'usdc_close', 'usdt_close',
       'btc_high', 'eth_high', 'usdc_high', 'usdt_high', 'btc_low', 'eth_low',
       'usdc_low', 'usdt_low', 'btc_open', 'eth_open', 'usdc_open',
       'usdt_open', 'btc_volume', 'eth_volume', 'usdc_volume', 'usdt_volume'],
      dtype='str')

In [20]:
crypto_daily_clean.shape

(2423, 21)

### Validation Result

After cleaning, the daily dataset:

- contains historical BTC, ETH, USDT and USDC market data;
- uses a consistent lowercase `snake_case` schema;
- stores dates as datetime values;
- stores market fields as numeric values;
- contains no duplicate dates;
- contains no unresolved missing values required for the analysis.

The cleaned dataset is saved separately from the raw source and is ready for event-level exploration.

In [21]:
crypto_daily_clean.to_csv("../data/processed/daily/crypto_daily_processed.csv", index=False)

## 3. Hourly Crypto-Market Data — Binance

The Binance datasets contain hourly BTC and ETH market data around the four selected historical events.

BTC and ETH are used to describe the **broader crypto-market environment** around each event. 

The cleaning stage standardizes:

- timestamps;
- numeric market fields;
- column names;
- chronological order.

The hourly datasets are also checked for duplicate timestamps and missing hourly observations.

In [22]:
crypto_hourly_raw = pd.read_csv(
    "../data/raw/hourly/binance/btc_celsius_hourly.csv"
)

crypto_hourly_raw.head()


,open_time,open,high,low,close,volume,close_time,quote_volume,trades,taker_buy_base,taker_buy_quote,ignore
0,1653782400000,29031.33,29045.34,28970.30,29005.47,786.45225,1653785999999,2.281502e+07,22692,279.37712,8.105066e+06,0
1,1653786000000,29005.46,29024.65,28900.00,28925.85,940.66592,1653789599999,2.724116e+07,24640,368.15723,1.066110e+07,0
2,1653789600000,28925.85,28925.86,28839.21,28904.13,1227.04778,1653793199999,3.544403e+07,27383,596.15271,1.721936e+07,0
3,1653793200000,28904.12,28997.21,28876.08,28984.32,719.15298,1653796799999,2.080649e+07,18690,351.82046,1.017869e+07,0
4,1653796800000,28984.31,28984.32,28914.96,28941.09,456.89978,1653800399999,1.322995e+07,16648,174.68137,5.058017e+06,0


In [23]:
crypto_hourly_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 721 entries, 0 to 720
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   open_time        721 non-null    int64  
 1   open             721 non-null    float64
 2   high             721 non-null    float64
 3   low              721 non-null    float64
 4   close            721 non-null    float64
 5   volume           721 non-null    float64
 6   close_time       721 non-null    int64  
 7   quote_volume     721 non-null    float64
 8   trades           721 non-null    int64  
 9   taker_buy_base   721 non-null    float64
 10  taker_buy_quote  721 non-null    float64
 11  ignore           721 non-null    int64  
dtypes: float64(8), int64(4)
memory usage: 67.7 KB


### Finding

The inspected Binance files use a consistent raw schema.

Market fields are stored as text and timestamps are stored as Unix milliseconds, so both require conversion before analysis.

The same cleaning and validation logic can therefore be applied consistently across all BTC and ETH event files.

In [24]:
def clean_binance_hourly(file_path):
    df = pd.read_csv(file_path)

    df["open_time"] = pd.to_datetime(
        df["open_time"],
        unit="ms"
    )

    selected_columns = [
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]

    df = df[selected_columns]

    numeric_columns = [
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]

    df[numeric_columns] = df[numeric_columns].astype(float)
    df = (df.sort_values("open_time").reset_index(drop=True))

    return df

In [25]:
def validate_hourly_data(df, timestamp_col):
    time_diff = (
        df[timestamp_col]
        .sort_values()
        .diff()
        .dropna()
    )

    gaps = time_diff[
        time_diff > pd.Timedelta(hours=1)
    ]

    missing_hours = (
        (gaps / pd.Timedelta(hours=1)) - 1
    ).sum()

    duplicate_timestamps = df.duplicated(
        subset=[timestamp_col]
    ).sum()

    hourly_frequency_ok = (
        duplicate_timestamps == 0
        and time_diff.eq(pd.Timedelta(hours=1)).all()
    )

    return {
        "rows": len(df),
        "missing_values": df.isna().sum().sum(),
        "duplicate_timestamps": duplicate_timestamps,
        "start_time": df[timestamp_col].min(),
        "end_time": df[timestamp_col].max(),
        "gap_count": len(gaps),
        "missing_hours": int(missing_hours),
        "hourly_frequency_ok": hourly_frequency_ok
    }

In [26]:
from pathlib import Path

hourly_files = Path("../data/raw/hourly/binance").glob("*.csv")

processed_binance = {}
validation_results = []

for file_path in hourly_files:
    df_clean = clean_binance_hourly(file_path)

    processed_binance[file_path.stem] = df_clean

    result = validate_hourly_data(df_clean, "open_time")
    result["file_name"] = file_path.name

    validation_results.append(result)

In [27]:
validation_summary = pd.DataFrame(validation_results)
validation_summary

,rows,missing_values,duplicate_timestamps,start_time,end_time,gap_count,missing_hours,hourly_frequency_ok,file_name
0,721,0,0,2022-05-29,2022-06-28,0,0,True,btc_celsius_hourly.csv
1,721,0,0,2024-10-23,2024-11-22,0,0,True,btc_election_hourly.csv
2,721,0,0,2022-04-25,2022-05-25,0,0,True,btc_luna_hourly.csv
3,720,0,0,2023-02-24,2023-03-26,1,1,False,btc_svb_hourly.csv
4,721,0,0,2022-05-29,2022-06-28,0,0,True,eth_celsius_hourly.csv
5,721,0,0,2024-10-23,2024-11-22,0,0,True,eth_election_hourly.csv
6,721,0,0,2022-04-25,2022-05-25,0,0,True,eth_luna_hourly.csv
7,720,0,0,2023-02-24,2023-03-26,1,1,False,eth_svb_hourly.csv


In [28]:
btc_svb = processed_binance["btc_svb_hourly"]

btc_svb[
    btc_svb["open_time"]
    .sort_values()
    .diff()
    > pd.Timedelta(hours=1)
]

,open_time,open,high,low,close,volume
685,2023-03-24 14:00:00,28079.99,28253.01,27835.0,27989.06,8983.24018


### Data Quality Finding — SVB Hourly Coverage

A small number of missing hourly BTC observations were identified in the wider SVB extraction window.

**Impact:**  
The gaps occur outside the core period used for the SVB event analysis.

**Decision:**  
No interpolation is applied. The missing observations are retained as a documented limitation because they do not affect the required analytical window.

In [29]:
output_dir = Path("../data/processed/hourly/binance")
output_dir.mkdir(parents=True, exist_ok=True)

for name, df in processed_binance.items():
    output_path = output_dir / f"{name}_processed.csv"

    df.to_csv(
        output_path,
        index=False
    )

## 4. Hourly Stablecoin Data — Coinbase

The Coinbase datasets contain hourly USDT/EUR and USDC/EUR market data around the same historical events.

In the modelled scenario:

- **USDT represents the operational balance after conversion from volatile cryptocurrencies.**
- **USDC is used as a comparison stablecoin.**

The comparison helps distinguish broader stablecoin-market movements from events that may be specific to one asset.

The cleaning stage standardizes timestamps and numeric market fields and checks each hourly series for duplicate or missing observations.

In [30]:
def clean_coinbase_hourly(file_path):
    df = pd.read_csv(file_path)

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        unit="s",
        utc=True
    )

    numeric_columns = [
        "low",
        "high",
        "open",
        "close",
        "volume"
    ]

    df[numeric_columns] = df[numeric_columns].astype(float)

    df = (
        df
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    return df

In [31]:
coinbase_files = Path("../data/raw/hourly/coinbase").glob("*.csv")

processed_coinbase = {}
coinbase_validation_results = []

for file_path in coinbase_files:
    df_clean = clean_coinbase_hourly(file_path)

    processed_coinbase[file_path.stem] = df_clean

    result = validate_hourly_data(
        df_clean,
        "timestamp"
    )

    result["file_name"] = file_path.name
    coinbase_validation_results.append(result)

In [32]:
coinbase_validation_summary = pd.DataFrame(
    coinbase_validation_results
)

coinbase_validation_summary

,rows,missing_values,duplicate_timestamps,start_time,end_time,gap_count,missing_hours,hourly_frequency_ok,file_name
0,721,0,0,2022-05-29 00:00:00+00:00,2022-06-28 00:00:00+00:00,0,0,True,usdc_celsius_hourly.csv
1,721,0,0,2024-10-23 00:00:00+00:00,2024-11-22 00:00:00+00:00,0,0,True,usdc_election_hourly.csv
2,721,0,0,2022-04-25 00:00:00+00:00,2022-05-25 00:00:00+00:00,0,0,True,usdc_luna_hourly.csv
3,714,0,0,2023-02-24 00:00:00+00:00,2023-03-26 00:00:00+00:00,2,7,False,usdc_svb_hourly.csv
4,721,0,0,2022-05-29 00:00:00+00:00,2022-06-28 00:00:00+00:00,0,0,True,usdt_celsius_hourly.csv
5,721,0,0,2024-10-23 00:00:00+00:00,2024-11-22 00:00:00+00:00,0,0,True,usdt_election_hourly.csv
6,721,0,0,2022-04-25 00:00:00+00:00,2022-05-25 00:00:00+00:00,0,0,True,usdt_luna_hourly.csv
7,716,0,0,2023-02-24 00:00:00+00:00,2023-03-26 00:00:00+00:00,1,5,False,usdt_svb_hourly.csv


In [33]:
usdc_svb = processed_coinbase["usdc_svb_hourly"]

usdc_svb[
    usdc_svb["timestamp"]
    .sort_values()
    .diff()
    > pd.Timedelta(hours=1)
]

,timestamp,low,high,open,close,volume
209,2023-03-04 23:00:00+00:00,0.9407,0.9409,0.9409,0.9409,3278.47
525,2023-03-18 04:00:00+00:00,0.9355,0.9358,0.9357,0.9356,5868.82


In [34]:
usdt_svb = processed_coinbase["usdt_svb_hourly"]

usdt_svb[
    usdt_svb["timestamp"]
    .sort_values()
    .diff()
    > pd.Timedelta(hours=1)
]

,timestamp,low,high,open,close,volume
209,2023-03-04 22:00:00+00:00,0.94093,0.94109,0.94107,0.94095,37996.28


### Data Quality Finding — SVB Stablecoin Coverage

Several missing hourly candles were identified in the wider SVB Coinbase extraction window.

**Impact:**  
The missing observations occur outside the core March event period required for the USDC analysis.

**Decision:**  
No interpolation is applied because the relevant event window remains fully covered.

The original gaps are preserved rather than creating synthetic market observations.

In [35]:
output_dir = Path("../data/processed/hourly/coinbase")
output_dir.mkdir(parents=True, exist_ok=True)

for name, df in processed_coinbase.items():
    output_path = output_dir / f"{name}_processed.csv"

    df.to_csv(
        output_path,
        index=False
    )

## 5. Hourly FX Benchmark — Dukascopy

Coinbase quotes USDT and USDC in EUR, while both stablecoins are designed to remain close to **$1**.

EUR/USD data is therefore used later as a benchmark to estimate their approximate USD value.

The purpose of the cleaning stage is only to standardize and validate the FX data. The stablecoin/FX combination is performed later in the event analyses.

The FX datasets are checked for:

- timestamp format;
- numeric market fields;
- duplicate observations;
- missing values;
- gaps between hourly observations.

In [36]:
df = pd.read_csv('../data/raw/fx/eur_usd_celsius_hourly.csv')
df

,Etc/UTC,Open,High,Low,Close,Volume
0,2022-05-29T21:00:00+00:00,1.07347,1.07355,1.07269,1.07328,1101320000
1,2022-05-29T22:00:00+00:00,1.07326,1.07360,1.07288,1.07301,3639680000
2,2022-05-29T23:00:00+00:00,1.07302,1.07314,1.07261,1.07290,3452330000
3,2022-05-30T00:00:00+00:00,1.07291,1.07346,1.07282,1.07298,4085780000
4,2022-05-30T01:00:00+00:00,1.07300,1.07528,1.07267,1.07483,5192430000
...,...,...,...,...,...,...
550,2022-06-29T19:00:00+00:00,1.04411,1.04447,1.04352,1.04409,3755840000
551,2022-06-29T20:00:00+00:00,1.04408,1.04437,1.04392,1.04398,2190900000
552,2022-06-29T21:00:00+00:00,1.04398,1.04443,1.04384,1.04406,1589870000
553,2022-06-29T22:00:00+00:00,1.04406,1.04454,1.04368,1.04439,10068980400


In [37]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 555 entries, 0 to 554
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Etc/UTC  555 non-null    str    
 1   Open     555 non-null    float64
 2   High     555 non-null    float64
 3   Low      555 non-null    float64
 4   Close    555 non-null    float64
 5   Volume   555 non-null    int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 39.7 KB


In [38]:
def clean_fx_hourly(file_path):
    df = pd.read_csv(file_path)

    df = df.rename(
        columns={
            "Etc/UTC": "timestamp",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Volume": "volume"
        }
    )

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        utc=True
    )

    df = (
        df
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    return df

In [39]:
def validate_fx_hourly(df):
    df = df.sort_values("timestamp").reset_index(drop=True)

    previous_timestamp = df["timestamp"].shift(1)
    time_diff = df["timestamp"] - previous_timestamp

    gaps = pd.DataFrame({
        "previous_timestamp": previous_timestamp,
        "timestamp": df["timestamp"],
        "time_diff": time_diff
    })

    gaps = gaps[
        gaps["time_diff"] > pd.Timedelta(hours=1)
    ].copy()

    gaps["is_weekend_gap"] = (
    (gaps["previous_timestamp"].dt.dayofweek == 4)
    &
    (gaps["timestamp"].dt.dayofweek == 6)
    &
    (gaps["time_diff"].between(
        pd.Timedelta(hours=48),
        pd.Timedelta(hours=50)
    ))
    )
    weekend_gaps = gaps[
        gaps["is_weekend_gap"]
    ]

    unexpected_gaps = gaps[
        ~gaps["is_weekend_gap"]
    ]

    duplicate_timestamps = df.duplicated(
        subset=["timestamp"]
    ).sum()

    missing_values = df.isna().sum().sum()

    fx_frequency_ok = (
        duplicate_timestamps == 0
        and len(unexpected_gaps) == 0
    )

    return {
        "rows": len(df),
        "missing_values": missing_values,
        "duplicate_timestamps": duplicate_timestamps,
        "start_time": df["timestamp"].min(),
        "end_time": df["timestamp"].max(),
        "weekend_gaps": len(weekend_gaps),
        "unexpected_gaps": len(unexpected_gaps),
        "fx_frequency_ok": fx_frequency_ok
    }

In [40]:
from pathlib import Path

fx_files = Path("../data/raw/fx").glob("*.csv")

processed_fx = {}
fx_validation_results = []

for file_path in fx_files:
    df_clean = clean_fx_hourly(file_path)

    processed_fx[file_path.stem] = df_clean

    result = validate_fx_hourly(df_clean)
    result["file_name"] = file_path.name

    fx_validation_results.append(result)

In [41]:
fx_validation_summary = pd.DataFrame(
    fx_validation_results
)

fx_validation_summary

,rows,missing_values,duplicate_timestamps,start_time,end_time,weekend_gaps,unexpected_gaps,fx_frequency_ok,file_name
0,555,0,0,2022-05-29 21:00:00+00:00,2022-06-29 23:00:00+00:00,4,0,True,eur_usd_celsius_hourly.csv
1,549,0,0,2024-10-23 00:00:00+00:00,2024-11-22 21:00:00+00:00,4,0,True,eur_usd_election_hourly.csv
2,552,0,0,2022-04-25 00:00:00+00:00,2022-05-25 23:00:00+00:00,4,0,True,eur_usd_luna_hourly.csv
3,502,0,0,2023-02-24 00:00:00+00:00,2023-03-24 20:00:00+00:00,4,0,True,eur_usd_svb_hourly.csv


### Data Quality Finding — FX Market Closures

The EUR/USD datasets contain no duplicate timestamps or unresolved missing values.

Longer gaps occur between Friday evening and Sunday evening observations.

These gaps are expected because the traditional foreign-exchange market does not trade continuously over the weekend, while crypto markets remain open.

The gaps are therefore treated as a difference in market trading schedules rather than random missing data.

No forward-fill is applied in the cleaning layer.

Where an event analysis requires an approximate USD value during a weekend, the most recent observable EUR/USD rate may be carried forward within that analysis. A separate flag is retained there to distinguish observed FX rates from carried-forward benchmark values.

In [42]:
output_dir = Path("../data/processed/fx")
output_dir.mkdir(parents=True, exist_ok=True)

for name, df in processed_fx.items():
    output_path = output_dir / f"{name}_processed.csv"

    df.to_csv(
        output_path,
        index=False
    )

## Data Cleaning Summary

The raw datasets from all project sources were cleaned and validated before event-level analysis.

The processed data now provides:

- standardized daily BTC, ETH, USDT and USDC market data;
- standardized hourly BTC and ETH data for broader market context;
- standardized hourly USDT data for the modelled operational balance;
- standardized hourly USDC data for comparison;
- standardized hourly EUR/USD data for later stablecoin/USD estimation.

Known data-quality limitations are documented rather than hidden or automatically interpolated.

In particular:

- small hourly gaps outside required event windows are retained without synthetic replacement;
- expected weekend FX market closures are preserved;
- any event-specific FX forward-fill is performed later in the EDA layer and clearly flagged.

The processed datasets are now ready for the four event studies.